# 📓 Text AI Module 2: Chain-of-Thought (CoT) Prompting & Reasoning Strategies
Welcome to Module 2 of Language Models! In this notebook, we explore **Chain-of-Thought (CoT)** prompting and reasoning architectures using a real open-weights Large Language Model (`Qwen/Qwen2.5-0.5B-Instruct` / `gpt2` fallback) [cite: 32, 33].

---

## 💡 1. Why Chain-of-Thought (CoT) Matters
Standard language model prompting maps a complex input directly to an answer [cite: 32]. While effective for direct fact retrieval, LLMs often fail at multi-step arithmetic, symbolic logic, and commonsense reasoning when required to output the answer immediately in one step [cite: 32].

**Chain-of-Thought (CoT) Prompting** (Wei et al., 2022) addresses this by encouraging the LLM to generate a sequence of intermediate natural language reasoning steps prior to giving the final answer [cite: 32].

### Key Prompting Paradigms:
1. **Standard Few-Shot:** Provides `(Question, Answer)` pairs without intermediate reasoning [cite: 32].
2. **Few-Shot CoT:** Provides `(Question, Reasoning Chain + Answer)` exemplar pairs [cite: 32].
3. **Zero-Shot CoT (Kojima et al., 2022):** Appends a simple trigger phrase like **`"Let's think step by step."`** to the user prompt, eliciting reasoning without needing pre-written examples [cite: 33]!


In [ ]:
# Install required packages if running in Colab/Jupyter
try:
    import transformers
except ImportError:
    !pip install -q transformers torch accelerate ipywidgets matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, Text, IntSlider

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load small open-weights LLM (Qwen2.5-0.5B or Qwen3.5-0.8B)
MODEL_ID = "Qwen/Qwen3.5-0.8B"
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading tokenizer and model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)

model.eval()
print(f"Successfully loaded {MODEL_ID} on {device}!")

## 🧪 2. Standard Prompting vs. Few-Shot CoT
Let's compare how the LLM performs on a multi-step word problem when prompted with **Standard Direct Exemplars** vs. **Few-Shot CoT Exemplars** [cite: 32].

In [ ]:
# Multi-step word problem
test_question = "A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there?"

# 1. Standard Few-Shot Prompt
standard_prompt = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: The answer is 11.

Q: """ + test_question + """
A: """

# 2. Few-Shot CoT Prompt
cot_prompt = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.

Q: """ + test_question + """
A: """

@torch.no_grad()
def generate_completion(prompt_text, max_tokens=60):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False  # Greedy for clear evaluation
    )
    # Decode only the generated response
    gen_text = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return gen_text.strip()

std_response = generate_completion(standard_prompt)
cot_response = generate_completion(cot_prompt)

print("=== Question ===")
print(test_question)
print("\n--- 1. Standard Prompt Output ---")
print(std_response)
print("\n--- 2. Few-Shot CoT Prompt Output ---")
print(cot_response)

## ⚙️ 3. Zero-Shot CoT: Two-Step Extraction Pipeline
In Zero-Shot CoT (Kojima et al., 2022), we don't supply any manual examples [cite: 33]. Instead, we use a **two-step extraction pipeline** [cite: 33]:

1. **Pass 1 (Reasoning Extraction):** Append `"Let's think step by step."` to the question and let the model generate its reasoning trajectory [cite: 33].
2. **Pass 2 (Answer Extraction):** Append the generated reasoning chain and trigger `"Therefore, the answer is:"` to isolate the final numerical/concise answer [cite: 33].

In [ ]:
@torch.no_grad()
def run_zero_shot_cot_pipeline(question, trigger="Let's think step by step."):
    # Step 1: Reasoning Extraction Pass
    step1_prompt = f"Q: {question}\nA: {trigger}"
    inputs1 = tokenizer(step1_prompt, return_tensors="pt").to(device)
    
    outputs1 = model.generate(
        **inputs1,
        max_new_tokens=80,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    reasoning_chain = tokenizer.decode(outputs1[0][inputs1.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    # Step 2: Answer Extraction Pass
    step2_prompt = f"{step1_prompt} {reasoning_chain}\nTherefore, the answer is:"
    inputs2 = tokenizer(step2_prompt, return_tensors="pt").to(device)
    
    outputs2 = model.generate(
        **inputs2,
        max_new_tokens=15,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    final_answer = tokenizer.decode(outputs2[0][inputs2.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    return step1_prompt, reasoning_chain, final_answer

math_problem = "On average Joe throws 25 punches per minute. A fight lasts 5 rounds of 3 minutes. How many punches did he throw?"
prompt_used, reasoning, answer = run_zero_shot_cot_pipeline(math_problem)

print("=== Zero-Shot CoT Two-Step Pipeline ===\n")
print(f"Initial Prompt : {prompt_used}")
print(f"Pass 1 (Extracted Reasoning):\n{reasoning}\n")
print(f"Pass 2 (Extracted Final Answer):\n{answer}")

## 🗺️ 4. Overview of Advanced Reasoning Paradigms
Modern LLM reasoning extends beyond simple linear chains into structured search and self-correcting graphs:

* **Tree-of-Thoughts (ToT):** Maintains a search tree over thought steps, enabling lookahead, backtracking, and beam search over intermediate reasoning states.
* **Self-Ask:** Prompts the model to explicitly output sub-questions, query external search/APIs for each sub-question, and synthesize answers.
* **Reflexion / Self-Correction:** Uses verbal feedback where the LLM evaluates its prior incorrect outputs and rewrites its reasoning trace.
* **Thought Preference Optimization (TPO):** Aligns reasoning models by optimizing user preferences directly over latent thought chains.

## 🎛️ 5. Interactive CoT Trigger Lab
Test different Zero-Shot CoT trigger phrases (e.g., `"Let's think step by step."` vs `"Take a deep breath and work on this problem step-by-step."` [cite: 33]) on custom reasoning problems!

In [ ]:
def interactive_cot_lab(question="A store has 20 shirts. They sell 5 shirts in the morning and receive 10 new shirts in the afternoon. How many shirts do they have now?",
                        trigger_phrase="Let's think step by step.",
                        max_tokens=70):
    prompt = f"Q: {question}\nA: {trigger_phrase}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    
    generated_reasoning = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    clear_output(wait=True)
    print(f"LLM Engine    : {MODEL_ID}")
    print(f"Trigger Used  : '{trigger_phrase}'")
    print("=" * 65)
    print(f"Full Prompt:\n{prompt}")
    print("-" * 65)
    print(f"Generated Reasoning Trace:\n{generated_reasoning}")
    print("=" * 65)

interact(interactive_cot_lab,
         question=Text(value="A store has 20 shirts. They sell 5 shirts in the morning and receive 10 new shirts in the afternoon. How many shirts do they have now?", description="Problem:"),
         trigger_phrase=Dropdown(
             options=[
                 "Let's think step by step.",
                 "Take a deep breath and work on this problem step-by-step.",
                 "Let me break this down mathematically:",
                 "Direct Answer:"
             ],
             value="Let's think step by step.",
             description="Trigger:"
         ),
         max_tokens=IntSlider(value=70, min=20, max=120, step=10, description="Max Tokens:"));